In [89]:
import os
from dotenv import load_dotenv

from openai import OpenAI
from qdrant_client import QdrantClient

In [90]:
load_dotenv()

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
qdrant_client = QdrantClient(host="localhost", port=6333)

COLLECTION_NAME = "renovation_docs"
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4.1-nano"

## Embedding query


In [91]:
def embed_query(query: str) -> list[float]:
    response = openai_client.embeddings.create(
        model=EMBED_MODEL,
        input=query,
    )
    return response.data[0].embedding

## Qdrant Retreival

In [92]:
def retrieve_candidates(query: str, top_k: int = 8):
    query_vector = embed_query(query)

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    return response.points

## Context

In [93]:
def build_context(points) -> str:
    parts = []

    for point in points:
        payload = point.payload or {}
        text = (payload.get("text") or "").strip()
        if text:
            parts.append(text)

    return "\n\n---\n\n".join(parts)

In [94]:
def rerank_chunks(question: str, candidates, top_n: int = 3):
    scored = []

    for point in candidates:
        payload = point.payload or {}
        chunk_text = (payload.get("text") or "").strip()

        if not chunk_text:
            continue

        rerank_prompt = f"""
Вопрос:
{question}

Фрагмент:
{chunk_text}

Оцени, насколько этот фрагмент помогает ответить на вопрос пользователя.

Верни только одно число от 0 до 10:
0 — фрагмент не помогает
10 — фрагмент максимально полезен
""".strip()

        response = openai_client.responses.create(
            model=CHAT_MODEL,
            input=[
                {
                    "role": "system",
                    "content": "Ты оцениваешь релевантность фрагмента для ответа на вопрос. Верни только число от 0 до 10."
                },
                {
                    "role": "user",
                    "content": rerank_prompt,
                },
            ],
            max_output_tokens=16,
            temperature=0,
        )

        raw_score = (response.output_text or "").strip()

        try:
            score = float(raw_score.replace(",", "."))
        except ValueError:
            score = 0.0

        scored.append((score, point))

    scored.sort(key=lambda item: item[0], reverse=True)
    return [point for score, point in scored[:top_n]]

## Prompt

In [95]:
SYSTEM_PROMPT = """
Ты — помощник компании по ремонту квартир.

Правила:
1. Отвечай только по переданному контексту.
2. Если информации достаточно — отвечай кратко, по делу и дружелюбно.
3. Если информации недостаточно, не выдумывай. Скажи по-человечески, что лучше уточнить детали у менеджера.
4. Если вопрос вообще не связан с ремонтом и услугами компании, коротко скажи, что ты помогаешь только по вопросам ремонта и услуг компании.
5. Не используй общие знания, если их нет в контексте.
6. Не упоминай базу знаний.
7. Отвечай по-русски, максимум 4-5 коротких предложений.
""".strip()

## Generator

In [104]:
OUT_OF_DOMAIN_MESSAGE = (
    "Я помогаю только с вопросами о ремонте и услугах компании."
)

MIN_RETRIEVAL_SCORE = 0.40

In [105]:
def answer_with_rag(question: str, top_k: int = 8, top_n: int = 3):
    candidates = retrieve_candidates(question, top_k=top_k)

    if not candidates:
        return OUT_OF_DOMAIN_MESSAGE, [], ""

    best_score = candidates[0].score or 0.0
    if best_score < MIN_RETRIEVAL_SCORE:
        return OUT_OF_DOMAIN_MESSAGE, candidates, ""

    best_points = rerank_chunks(question, candidates, top_n=top_n)

    if not best_points:
        return "Не могу сейчас ответить точно. Лучше уточнить детали у менеджера.", [], ""

    context = build_context(best_points)

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}

Ответь по правилам.
Если данных мало, скажи, что лучше уточнить детали у менеджера.
Если вопрос не по теме ремонта и услуг компании, скажи, что ты помогаешь только по этим вопросам.
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, best_points, context

In [97]:
def answer_with_baseline(question: str, top_k: int = 5):
    points = retrieve_candidates(question, top_k=top_k)
    context = build_context(points[:top_k])

    user_prompt = f"""
Вопрос пользователя:
{question}

Контекст:
{context}

Ответь по правилам.
""".strip()

    response = openai_client.responses.create(
        model=CHAT_MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=140,
        temperature=0,
    )

    answer = (response.output_text or "").strip()
    return answer, points[:top_k], context

# Tests

In [100]:
def print_points(points, max_chars: int = 500):
    for i, point in enumerate(points, start=1):
        payload = point.payload or {}
        print("=" * 100)
        print(f"RESULT #{i}")
        print(f"score: {getattr(point, 'score', None)}")
        print(f"title: {payload.get('title')}")
        print(f"url: {payload.get('url')}")
        print()
        print((payload.get("text") or "")[:max_chars])
        print()

In [101]:
def compare_answers(question: str, baseline_top_k: int = 5, rerank_top_k: int = 8, rerank_top_n: int = 3):
    print("#" * 120)
    print("QUESTION:", question)
    print()

    baseline_answer, baseline_points, _ = answer_with_baseline(question, top_k=baseline_top_k)
    print("BASELINE ANSWER:")
    print(baseline_answer)
    print()

    rerank_answer, rerank_points, _ = answer_with_rag(question, top_k=rerank_top_k, top_n=rerank_top_n)
    print("RERANK ANSWER:")
    print(rerank_answer)
    print()

    print("BASELINE POINTS:")
    print_points(baseline_points, max_chars=350)

    print("RERANK POINTS:")
    print_points(rerank_points, max_chars=350)

In [102]:
test_questions = [
    "Сколько стоит сделать ремонт на кухне?",
    "Мне нужен ремонт квартиры 40 кв.м. сколько будет примерно стоить?",
    "Зависит ли расположение квартиры на стоимость работ?",
    "Что влияет на стоимость ремонта квартиры?",
    "Делаете ли вы дизайнерский ремонт?",
    "Есть ли гарантия на ремонт?",
    "Вы делаете ремонт складских помещений?",
    "Работаете ли вы с коммерческими помещениями?",
    "Объясни квантовую механику",
    "Какая погода в Москве?",
]

In [103]:
for q in test_questions:
    compare_answers(q)

########################################################################################################################
QUESTION: Сколько стоит сделать ремонт на кухне?

BASELINE ANSWER:
Стоимость ремонта кухни зависит от объема работ. Например, демонтаж кухонной мойки стоит 670 рублей за штуку. Для полного ремонта кухни лучше уточнить у менеджера детали и состав работ.

RERANK ANSWER:
Стоимость ремонта кухни зависит от объема работ и выбранных материалов. Для точной стоимости лучше уточнить детали у менеджера. Могу помочь с ценами на демонтаж кухонной мойки — 670 рублей за штуку.

BASELINE POINTS:
RESULT #1
score: 0.5462564
title: Демонтаж кухонной мойки (без сохранения)
url: https://sk-family.ru/db.json#demontazh_kuhonnoy_moyki_bez_sohraneniya

Категория: Демонтажные работы
Раздел: Сантехника
Услуга: Демонтаж кухонной мойки (без сохранения)
Единица измерения: шт.
Цена: 670.0

RESULT #2
score: 0.5372038
title: Компания по ремонту квартир в Москве
url: https://sk-family.ru/

# Ремонт 

In [119]:
answer, _, _ = answer_with_rag("Какие сроки работ у вас? от и до! и есть ли гарантия?")
print(answer)

Сроки выполнения работ у нас от 2 до 4 месяцев, в зависимости от объема и сложности проекта. Гарантия на выполненные работы — до 3 лет. Для точных сроков и условий лучше уточнить у менеджера.


In [113]:
compare_answers("мне нужен демантаж всего в на кухне 12 метровБ сколко стоит примерно?")

########################################################################################################################
QUESTION: мне нужен демантаж всего в на кухне 12 метровБ сколко стоит примерно?

BASELINE ANSWER:
Для точной стоимости лучше уточнить у менеджера, так как зависит от объема и сложности работ. Обычно цена за демонтаж в кухне — около 200 рублей за кв.м. Если площадь 12 м², примерно получится 2400 рублей.

RERANK ANSWER:
Для точной стоимости лучше уточнить у менеджера, так как зависит от объема и сложности работ. Обычно цена за кв.м. — 200 рублей.

BASELINE POINTS:
RESULT #1
score: 0.47467715
title: Снятие многослойных / трудноотделяемых обоев
url: https://sk-family.ru/db.json#snyatie_mnogosloynyh_trudnootdelyaemyh_oboev

Категория: Демонтажные работы
Раздел: Стены
Услуга: Снятие многослойных / трудноотделяемых обоев
Единица измерения: кв.м
Цена: 200.0

RESULT #2
score: 0.4719329
title: Снятие многослойных / трудноотделяемых обоев (потолок)
url: https://sk-family.ru/db.